<div style="background: linear-gradient(90deg, #e0eafc 0%, #cfdef3 100%); border-radius: 12px; padding: 32px 24px; margin-bottom: 24px; box-shadow: 0 2px 8px rgba(0,0,0,0.07);">

# 🌟 Agentes con Memoria Persistente en Lakebase

Bienvenido a este notebook interactivo, donde aprenderás a construir y desplegar un **agente inteligente con memoria persistente** usando las tecnologías más avanzadas de Databricks:

- **LangGraph**: Orquestación flexible de agentes conversacionales.
- **Lakebase**: Almacenamiento de memoria de conversación respaldado por PostgreSQL.
- **MLflow**: Registro, versionado y despliegue de modelos.
- **Unity Catalog**: Organización y seguridad de recursos.
- **Model Serving**: Despliegue en producción con endpoints de LLM.

---

## 🚀 ¿Qué vas a lograr aquí?

- Crear un agente que **recuerda conversaciones** usando IDs de hilo.
- Optimizar la gestión de conexiones para alta concurrencia.
- Registrar y desplegar el modelo en MLflow y Unity Catalog.
- Probar la memoria del agente y su capacidad de streaming.
- Preparar tu entorno para producción siguiendo las mejores prácticas de Databricks FE.

---

> **Sigue los pasos del notebook para descubrir cómo los agentes con memoria pueden transformar la experiencia conversacional en tu Lakehouse.**

</div>

## ⚙️ Configuración Actual del Notebook

### 📌 Valores que DEBES revisar/cambiar:

 Componente | Ubicación | Valor Actual | ¿Necesita cambio? |
------------|-----------|--------------|--------------------|
 **Lakebase Instance** | Cell 5, línea 42 | `dv-agents-memory` | ✅ Sí, si usas otra instancia |
 **LLM Endpoint** | Cell 5, línea 38 | `databricks-claude-3-7-sonnet` | ✅ Sí, si no está disponible |
 **UC Catalog** | Cell 13 | `users` | ✅ Sí, cambia a tu Catalogo  |
 **UC Schema** | Cell 13 | `daniel_vargas` | ✅ Sí, cambia a tu schema |
 **Model Name** | Cell 13 | `stateful_lakebase_agent` | 🟡 Opcional (pero recomendado) |

---

# Agente LangGraph con memoria persistente en Lakebase

Este notebook demuestra cómo construir y desplegar un **agente con estado** usando:
* **LangGraph** - Para la orquestación del agente
* **Lakebase** - Para memoria persistente de conversación (respaldado por PostgreSQL)
* **MLflow** - Para seguimiento y despliegue del modelo
* **Unity Catalog** - Para el registro de modelos
* **Databricks Model Serving** - Para despliegue en producción

## Características clave
* ✅ **Memoria de conversación**: El agente recuerda interacciones pasadas usando IDs de hilo
* ✅ **Pool de conexiones optimizado**: Maneja alta concurrencia sin timeouts
* ✅ **Listo para producción**: Totalmente rastreado, evaluado y desplegado
* ✅ **Escalable**: Patrón singleton para uso eficiente de recursos

## Arquitectura

Entrada del usuario → Agente LangGraph → LLM (Claude 3.7 Sonnet)
                ↓
        Lakebase (PostgreSQL)
        - Almacena el estado de la conversación
        - Memoria basada en hilos
        - Pool de conexiones

In [0]:
# Install compatible versions of all package
print("📦 Installing dependencies (this takes 2-3 minutes)...\n")

# Install packages with compatible versions
%pip install -U --quiet \
  'databricks-langchain[memory]>=0.16.0' \
  'databricks-ai-bridge>=0.15.0' \
  'databricks-vectorsearch>=0.50' \
  'mlflow>=3.0.0' \
  'langgraph>=1.0.0' \
  'databricks-agents>=0.9.0'

print("\n✅ Installation complete!")
print("🔄 Restarting Python to load new packages...\n")

dbutils.library.restartPython()

In [0]:
%%writefile agent.py
import logging
import os
import uuid
from typing import Annotated, Any, Generator, Optional, Sequence, TypedDict

import mlflow
from databricks_langchain import (
    ChatDatabricks,
    UCFunctionToolkit,
    CheckpointSaver,
)
from databricks.sdk import WorkspaceClient
from langchain_core.messages import AIMessage, AIMessageChunk, AnyMessage
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
)

logger = logging.getLogger(__name__)
logging.basicConfig(level=os.getenv("LOG_LEVEL", "INFO"))

############################################
# Define your LLM endpoint and system prompt
############################################
LLM_ENDPOINT_NAME = "databricks-claude-3-7-sonnet"
SYSTEM_PROMPT = "You are a helpful assistant. Use the available tools to answer questions."

############################################
# Lakebase configuration
############################################
LAKEBASE_INSTANCE_NAME = "dv-agents-memory"

# Optimized connection pool settings for psycopg ConnectionPool
LAKEBASE_POOL_CONFIG = {
    "max_size": 50,  # Maximum pool size (increased from default ~10)
    "min_size": 2,   # Minimum pool size
    "timeout": 120.0,  # Timeout for getting connection (2 minutes)
    "max_waiting": 10,  # Allow up to 10 requests to queue for connections
    "max_lifetime": 1800.0,  # Maximum connection lifetime (30 minutes)
    "max_idle": 60.0,  # Release idle connections after 1 minute
    "reconnect_timeout": 60.0,  # Time to attempt reconnection
}

###############################################################################
## Define tools for your agent
###############################################################################
tools = []

# Example UC tools; add your own as needed
UC_TOOL_NAMES: list[str] = []
if UC_TOOL_NAMES:
    uc_toolkit = UCFunctionToolkit(function_names=UC_TOOL_NAMES)
    tools.extend(uc_toolkit.tools)

# Vector search tools
VECTOR_SEARCH_TOOLS = []
tools.extend(VECTOR_SEARCH_TOOLS)

#####################
## Define agent logic
#####################

class AgentState(TypedDict):
    messages: Annotated[Sequence[AnyMessage], add_messages]
    custom_inputs: Optional[dict[str, Any]]
    custom_outputs: Optional[dict[str, Any]]

# Global checkpointer instance to reuse connections (singleton pattern)
_checkpointer_instance = None

def get_checkpointer():
    """Get or create a singleton CheckpointSaver instance."""
    global _checkpointer_instance
    if _checkpointer_instance is None:
        _checkpointer_instance = CheckpointSaver(
            instance_name=LAKEBASE_INSTANCE_NAME,
            **LAKEBASE_POOL_CONFIG
        )
    return _checkpointer_instance

class LangGraphResponsesAgent(ResponsesAgent):
    """Stateful agent using ResponsesAgent with pooled Lakebase checkpointing."""

    def __init__(self, lakebase_config: dict[str, Any]):
        self.workspace_client = WorkspaceClient()
        self.model = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)
        self.system_prompt = SYSTEM_PROMPT
        self.model_with_tools = self.model.bind_tools(tools) if tools else self.model

    def _create_graph(self, checkpointer: Any):
        def should_continue(state: AgentState):
            messages = state["messages"]
            last_message = messages[-1]
            if isinstance(last_message, AIMessage) and last_message.tool_calls:
                return "continue"
            return "end"

        preprocessor = (
            RunnableLambda(lambda state: [{"role": "system", "content": self.system_prompt}] + state["messages"])
            if self.system_prompt
            else RunnableLambda(lambda state: state["messages"])
        )
        model_runnable = preprocessor | self.model_with_tools

        def call_model(state: AgentState, config: RunnableConfig):
            response = model_runnable.invoke(state, config)
            return {"messages": [response]}

        workflow = StateGraph(AgentState)
        workflow.add_node("agent", RunnableLambda(call_model))

        if tools:
            workflow.add_node("tools", ToolNode(tools))
            workflow.add_conditional_edges("agent", should_continue, {"continue": "tools", "end": END})
            workflow.add_edge("tools", "agent")
        else:
            workflow.add_edge("agent", END)

        workflow.set_entry_point("agent")
        return workflow.compile(checkpointer=checkpointer)

    def _get_or_create_thread_id(self, request: ResponsesAgentRequest) -> str:
        """Get thread_id from request or create a new one.

        Priority:
        1. Use thread_id from custom_inputs if present
        2. Use conversation_id from chat context if available
        3. Generate a new UUID

        Returns:
            thread_id: The thread identifier to use for this conversation
        """
        ci = dict(request.custom_inputs or {})

        if "thread_id" in ci:
            return ci["thread_id"]

        # using conversation id from chat context as thread id
        if request.context and getattr(request.context, "conversation_id", None):
            return request.context.conversation_id

        # Generate new thread_id
        return str(uuid.uuid4())

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    def predict_stream(
        self, request: ResponsesAgentRequest
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        thread_id = self._get_or_create_thread_id(request)
        ci = dict(request.custom_inputs or {})
        ci["thread_id"] = thread_id
        request.custom_inputs = ci

        # Convert incoming Responses messages to ChatCompletions format
        cc_msgs = self.prep_msgs_for_cc_llm([i.model_dump() for i in request.input])
        langchain_msgs = cc_msgs
        checkpoint_config = {"configurable": {"thread_id": thread_id}}

        # Use singleton checkpointer instance to reuse connections
        checkpointer = get_checkpointer()
        graph = self._create_graph(checkpointer)

        for event in graph.stream(
            {"messages": langchain_msgs},
            checkpoint_config,
            stream_mode=["updates", "messages"],
        ):
            if event[0] == "updates":
                for node_data in event[1].values():
                    if len(node_data.get("messages", [])) > 0:
                        yield from output_to_responses_items_stream(node_data["messages"])
            elif event[0] == "messages":
                try:
                    chunk = event[1][0]
                    if isinstance(chunk, AIMessageChunk) and chunk.content:
                        yield ResponsesAgentStreamEvent(
                            **self.create_text_delta(delta=chunk.content, item_id=chunk.id),
                        )
                except Exception as exc:
                    logger.error("Error streaming chunk: %s", exc)


# ----- Export model -----
mlflow.langchain.autolog()
AGENT = LangGraphResponsesAgent(LAKEBASE_INSTANCE_NAME)
mlflow.models.set_model(AGENT)

In [0]:
from agent import AGENT

print("🧪 Test 1: Create new conversation thread\n")

# First message - establish context
result1 = AGENT.predict({
    "input": [{"role": "user", "content": "I am working on stateful agents with Lakebase"}]
})

# Extract thread_id from response
thread_id = result1.custom_outputs["thread_id"]
response_text = result1.output[0].content[0]['text']  # Fixed: access as dict, not attribute

print(f"✅ Response received!")
print(f"Thread ID: {thread_id}")
print(f"\nAgent: {response_text[:200]}...")
print(f"\n💾 Conversation state saved to Lakebase")

In [0]:
print("🧪 Test 2: Verify conversation memory\n")
print(f"Using thread_id: {thread_id}\n")

# Second message - should remember context
result2 = AGENT.predict({
    "input": [{"role": "user", "content": "What am I working on?"}],
    "custom_inputs": {"thread_id": thread_id}
})

response_text = result2.output[0].content[0]['text']  # Fixed: access as dict, not attribute

print(f"✅ Response received!")
print(f"\nAgent: {response_text}")

# Check if agent remembered the context
if "stateful" in response_text.lower() or "lakebase" in response_text.lower():
    print("\n🎉 SUCCESS - Agent remembered the conversation!")
    print("   Memory persistence via Lakebase is working correctly")
else:
    print("\n⚠️  Agent response doesn't clearly reference previous context")
    print("   Memory may need verification")

In [0]:
print("🧪 Test 3: Test streaming with memory\n")
print(f"Using thread_id: {thread_id}\n")

print("Agent (streaming): ", end="")

for event in AGENT.predict_stream({
    "input": [{"role": "user", "content": "Summarize what we've discussed"}],
    "custom_inputs": {"thread_id": thread_id}
}):
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)

print("\n\n✅ Streaming test complete!")
print("\n🎯 All tests passed - Agent is ready for deployment")

In [0]:
# Restart Python to clear all Lakebase connections from test runs
print("🔄 Restarting Python to clear connection pool...\n")
print("This ensures MLflow logging has clean access to Lakebase\n")

dbutils.library.restartPython()

# Registrar el modelo en MLflow es importante porque permite:

- Guardar y versionar el modelo de manera segura y organizada.
- Rastrear la evolución del modelo a lo largo del tiempo.
- Facilitar la colaboración entre equipos.
- Desplegar el modelo fácilmente en producción.
- Cumplir con políticas de retención y organización de recursos en Databricks.

In [0]:
import mlflow
from mlflow.models.resources import DatabricksLakebase

print("📦 Logging agent to MLflow...\n")

# Define input example for model validation
input_example = {
    "input": [{"role": "user", "content": "Hello! Can you help me with data analysis?"}]
}

# Log the agent with Lakebase resource declaration
with mlflow.start_run() as run:
    logged_agent_info = mlflow.pyfunc.log_model(
        python_model="agent.py",
        artifact_path="agent",
        input_example=input_example,
        resources=[DatabricksLakebase(database_instance_name="dv-agents-memory")],  # Fixed: correct parameter name
    )

print(f"✅ Agent logged successfully!")
print(f"   Run ID: {run.info.run_id}")
print(f"   Model URI: {logged_agent_info.model_uri}")
print(f"   Lakebase Resource: dv-agents-memory")
print(f"\n🔗 View in MLflow: {mlflow.get_experiment(run.info.experiment_id).name}")

## Registrando nuestro agente a Unity Catalog

In [0]:
# Unity Catalog registration configuration
# Based on workspace policy: using 'users' catalog with personal schema

catalog = "users"  # Required: use 'users' catalog per workspace policy
schema = "daniel_vargas"  # Your personal schema (matches your username)
model_name = "stateful_lakebase_agent"  # Descriptive model name

print(f"📍 UC Model Registration Config:")
print(f"   Catalog: {catalog}")
print(f"   Schema: {schema}")
print(f"   Model: {model_name}")
print(f"\n   Full model name: {catalog}.{schema}.{model_name}")

In [0]:
import mlflow
mlflow.set_registry_uri("databricks-uc")

UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

print(f"📚 Registering model to Unity Catalog...")
print(f"   Target: {UC_MODEL_NAME}")
print(f"   Per workspace policy: Using 'users' catalog with personal schema\n")

# Register the model to UC
uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info.model_uri, 
    name=UC_MODEL_NAME
)

print(f"✅ Model registered successfully!")
print(f"   Name: {UC_MODEL_NAME}")
print(f"   Version: {uc_registered_model_info.version}")
print(f"\n🔗 View in UC: Catalog Explorer → {catalog} → {schema} → {model_name}")

In [0]:
import mlflow

# Search for all versions of the model in Unity Catalog
from mlflow.tracking import MlflowClient
client = MlflowClient()

try:
    # For Unity Catalog, search for model versions
    versions = client.search_model_versions(f"name='{model_name}'")
    
    if versions:
        # Get the latest version (highest version number)
        latest_version = max(versions, key=lambda v: int(v.version))
        model_version = latest_version.version
        
        model_info = client.get_model_version(model_name, model_version)
        print(model_info)
    else:
        print(f"No versions found for model: {model_name}")
        print("Please run Cell 11 (Register to Unity Catalog) first.")
except Exception as e:
    print(f"Error: {e}")
    print("Please run Cell 11 (Register to Unity Catalog) first.")


## 🧪 Probar tu agente en Review Apps de Databricks

Una vez que hayas desplegado tu agente en Model Serving (ver la celda siguiente), puedes probarlo fácilmente usando **Review Apps** de Databricks:

1. **Despliega el agente** usando la celda de abajo ("Deploy Agent").
2. Cuando el despliegue termine, aparecerá un enlace a la Review App en la salida de la celda.
3. Haz clic en el enlace para abrir la Review App. Allí podrás interactuar con tu agente, enviar mensajes y verificar que la memoria persistente funciona correctamente.
4. Puedes probar diferentes hilos de conversación, enviar preguntas y ver cómo el agente recuerda el contexto gracias a Lakebase.

> **Tip:** Si el endpoint aún está iniciando, espera unos minutos y vuelve a cargar la Review App.

---

Continúa con la celda siguiente para desplegar tu agente y obtener el enlace de prueba.

In [0]:
from databricks import agents
from mlflow.tracking import MlflowClient

UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

print(f"🚀 Deploying agent to Model Serving...")
print(f"   Model: {UC_MODEL_NAME}\n")

# Get the latest version of the model
client = MlflowClient()
versions = client.search_model_versions(f"name='{UC_MODEL_NAME}'")

if versions:
    # Get the latest version (highest version number)
    latest_version = max(versions, key=lambda v: int(v.version))
    model_version = latest_version.version
    
    print(f"   Version: {model_version}\n")
    
    # Deploy the agent
    deployment = agents.deploy(
        UC_MODEL_NAME, 
        model_version, 
        tags={"endpointSource": "stateful-agent-v2"},
        deploy_feedback_model=False
    )
    
    print(f"\n✅ Deployment initiated!")
    print(f"\n🕗 This can take up to 15 minutes...")
    print(f"\n🔗 Access your agent:")
    print(f"   Review App: Check the output above for the link")
    print(f"   Endpoint Status: ML → Serving Endpoints → {deployment.endpoint_name if hasattr(deployment, 'endpoint_name') else 'agents_' + UC_MODEL_NAME.replace('.', '-')}")
else:
    print(f"❌ No versions found for model: {UC_MODEL_NAME}")
    print("Please run Cell 11 (Register to Unity Catalog) first.")